# 2. Discovery by disagreement

Three detectors read the same practice field: P (catalog), D (difference), A (anomaly ensemble). The agreement pattern of each candidate is kept and compared with the hidden truth.

In [1]:
import os, sys, json, math, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(ROOT, "backend"))
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
for f in os.listdir(os.path.join(ROOT, "scripts", "fonts")):
    font_manager.fontManager.addfont(os.path.join(ROOT, "scripts", "fonts", f))
plt.rcParams.update({"font.family": "EB Garamond", "font.size": 12, "figure.facecolor": "#f5f0e6", "axes.facecolor": "#fbf8f2"})
import core, sky, pipeline as P
import investigate as I

In [2]:
spec = sky.make_spec(3)
E, truth, cat = sky.render(spec)
out = I.investigate(E, cat, center=spec["center"], truth=truth, spec=spec, meta=dict(mode="sim", key="demo"))
for c in out["candidates"]:
    p = c["post"]
    print(c["id"], c["methods"].ljust(3), "truth:", (c["truth_sub"] or c["truth"]).ljust(8), "verdict:", ("unknown" if p["unknown"] else p["verdict"]).ljust(9), "|", c["dossier"]["status"])

C01 PDA truth: periodic verdict: variable  | Worth further investigation. Not a confirmed discovery.
C02 PDA truth: flare    verdict: transient | Worth further investigation. Not a confirmed discovery.
C03 PA  truth: appear   verdict: transient | Worth further investigation. Not a confirmed discovery.
C04 PA  truth: hot      verdict: artefact  | Most consistent with an instrumental artefact.
C05 PDA truth: ghost    verdict: artefact  | Most consistent with an instrumental artefact.
C06 A   truth: glitch   verdict: artefact  | Most consistent with an instrumental artefact.
C07 P   truth: mover    verdict: moving    | Worth further investigation. Not a confirmed discovery.


## Why they disagree

Each pattern has a measured meaning and asks for a test.

In [3]:
for c in out["candidates"]:
    d = c["dossier"]["disagreement"]
    print(c["id"], d["pattern"], "-", d["headline"])

C01 PDA - All three methods agree.
C02 PDA - All three methods agree.
C03 PA - Catalog and anomaly agree; no fair pair shows it.
C04 PA - Catalog and anomaly agree; no fair pair shows it.
C05 PDA - All three methods agree.
C06 A - Only the anomaly model sees it.
C07 P - Only the catalog method sees it.


## The crosstalk case: agreement is not proof

In [4]:
g = next(c for c in out["candidates"] if c["truth_sub"] == "ghost")
for t in g["dossier"]["tests"]:
    print(t["outcome"].ljust(11), t["question"], "|", t["result"])

supports    Would a real source show in the other array at the same moment? | The partner does not see it.
neutral     Is it fixed to detector pixels? | The detector positions spread over 1217.4 pixels.
supports    Is it a crosstalk copy of a bright star? | Yes. A bright star sits 128 columns to the right on the same row in f3_e1_d4_s1.
neutral     Is it sharper than the optics allow? | Width ratio 0.96.
neutral     Does it touch flagged pixels? | 0 percent of exposures.
supports    Is it seen by one array only? | 86 percent in one array.


## The difference microscope

Spots above 5 sigma in each layer of correction for every fair pair.

In [5]:
for L in out["field"]["microscope"]:
    print("visits", L["q1"], L["q2"], "array", L["det"], "fair" if L["fair"] else "NOT FAIR", [(x["key"], x["count"]) for x in L["layers"]])

visits 1 2 array 1 fair [('raw', 28), ('wcs', 31), ('anchor', 10), ('background', 10), ('noise', 2), ('coadd', 2)]
visits 1 2 array 2 fair [('raw', 27), ('wcs', 27), ('anchor', 10), ('background', 12), ('noise', 1), ('coadd', 1)]
visits 1 2 array 3 fair [('raw', 29), ('wcs', 15), ('anchor', 12), ('background', 12), ('noise', 1), ('coadd', 2)]
visits 1 2 array 5 fair [('raw', 29), ('wcs', 2), ('anchor', 1), ('background', 1), ('noise', 1), ('coadd', 1)]
visits 1 2 array 6 fair [('raw', 30), ('wcs', 16), ('anchor', 3), ('background', 3), ('noise', 1), ('coadd', 1)]
visits 3 4 array 1 fair [('raw', 29), ('wcs', 29), ('anchor', 7), ('background', 7), ('noise', 2), ('coadd', 2)]
visits 3 4 array 2 fair [('raw', 29), ('wcs', 25), ('anchor', 11), ('background', 13), ('noise', 1), ('coadd', 0)]
visits 3 4 array 3 fair [('raw', 29), ('wcs', 29), ('anchor', 11), ('background', 11), ('noise', 1), ('coadd', 0)]
visits 3 4 array 4 fair [('raw', 29), ('wcs', 29), ('anchor', 5), ('background', 5), ('

## The report of the moving candidate

Every sentence carries evidence references.

In [6]:
m = next(c for c in out["candidates"] if c["truth"] == "moving")
for s in m["dossier"]["report"]:
    print(s["text"], s["refs"])
print()
for e in m["dossier"]["evidence"]:
    print(e["id"], e["what"], ":", e["value"])

C07 was flagged by the catalog method, and not by the difference method, the anomaly method. ['E1']
It shifted by 0.41 pixels over 548 days, and a fixed position fits worse. ['E2', 'E3']
It passed 4 tests that tried to break the leading explanation. ['E4']
It survives 9 of 11 reasonable changes to the analysis. ['E5']
The most likely explanation is moving object at 98 percent. ['E6']

E1 Methods that flagged it : P
E2 Motion : 0.270 plus or minus 0.044 pixels per year (1.66 arcseconds per year)
E3 Fixed position against motion : delta BIC 33.7
E4 Challenges survived : 4 support, 0 argue against, 1 neutral or not run
E5 Robustness : 9 of 11 analysis choices keep it above the threshold
E6 Hypothesis probabilities : H1 98 percent, H2 0 percent, H3 0 percent, H4 0 percent, H5 1 percent
